# Detached progress monitor

**Inputs:** v3 public reports. **Outputs:** refreshed progress PNG/JSON every 60 seconds. **Runtime:** until the controller completes or fails; runs in detached tmux. **Recovery:** Run All with `AHRV3_MONITOR=1` to resume. Default Run All makes one snapshot. No training and no scientific checkpoint writes.

In [ ]:
from pathlib import Path
import nbformat
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run within AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
        exec(compile(cell.source,'v3-shared:export','exec'),globals())

v3_import_exports(PROJECT/'experiments/ahr_jepa_v3/progress.ipynb',globals())
import time,os


In [ ]:
while True:
    snapshot=v3_refresh_progress(PROJECT)
    print(snapshot['generated_at_utc'],(snapshot['pipeline'] or {}).get('status','NOT_LAUNCHED'),flush=True)
    if os.environ.get('AHRV3_MONITOR')!='1':break
    if (snapshot['pipeline'] or {}).get('status') in ('COMPLETE','FAILED','INTERRUPTED_CHILD_ACTIVE'):break
    time.sleep(60)
